# Notebook 4 — Résultats, visualisations et tableaux pour le mémoire

## Position dans le pipeline

`NB1 → NB2 → NB3 → **NB4 (vous êtes ici)**`  
Guide : `00_PIPELINE_PAS_A_PAS.md`

## Objectif

Produire les **tableaux et figures** du chapitre Résultats à partir des sorties déjà calculées.

## Entrées attendues (Notebooks 2–3)

- `data/processed/opensensemap_qc_metrics_station_variable.csv`
- `data/processed/opensensemap_trust_scores_series.csv`
- `data/processed/opensensemap_trust_scores_stations.csv`
- `data/processed/opensensemap_fit_for_purpose_summary.csv`
- `data/processed/opensensemap_qc_parameters.json`
- `data/processed/opensensemap_trust_parameters.json`

## Sorties

- `reports/tables/*.csv`
- `reports/figures/*.png` (distributions, classes FFP, profils, cartes, diagnostics)
- `reports/manifest_memoires_exports.csv`

Ce notebook **ne recalcule pas** le QC ni les scores : il **présente** les résultats pour le mémoire.

> Les scores de confiance sont relatifs et contextuels. Ils ne constituent pas une certification d’exactitude absolue.


## 1. Chargement des résultats

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"

for folder in [REPORTS_DIR, TABLES_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

METRICS_CSV = PROCESSED_DIR / "opensensemap_qc_metrics_station_variable.csv"
TRUST_SERIES_CSV = PROCESSED_DIR / "opensensemap_trust_scores_series.csv"
TRUST_STATIONS_CSV = PROCESSED_DIR / "opensensemap_trust_scores_stations.csv"
FFP_SUMMARY_CSV = PROCESSED_DIR / "opensensemap_fit_for_purpose_summary.csv"
QC_PARAMS_JSON = PROCESSED_DIR / "opensensemap_qc_parameters.json"
TRUST_PARAMS_JSON = PROCESSED_DIR / "opensensemap_trust_parameters.json"
MANIFEST_CSV = REPORTS_DIR / "manifest_memoires_exports.csv"

required = [
    METRICS_CSV, TRUST_SERIES_CSV, TRUST_STATIONS_CSV,
    FFP_SUMMARY_CSV, QC_PARAMS_JSON, TRUST_PARAMS_JSON,
]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Fichiers manquants — exécuter NB2/NB3 (ou implement_trust_framework.py) :\n"
        + "\n".join(f"- {p}" for p in missing)
    )

metrics = pd.read_csv(METRICS_CSV)
trust_series = pd.read_csv(TRUST_SERIES_CSV)
trust_stations = pd.read_csv(TRUST_STATIONS_CSV)
ffp_summary = pd.read_csv(FFP_SUMMARY_CSV)

with open(TRUST_PARAMS_JSON, encoding="utf-8") as f:
    trust_params = json.load(f)
with open(QC_PARAMS_JSON, encoding="utf-8") as f:
    qc_params = json.load(f)

print("Séries (métriques) :", len(metrics))
print("Séries (confiance) :", len(trust_series))
print("Stations           :", len(trust_stations))
print("Sorties            :", TABLES_DIR, "|", FIGURES_DIR)
display(trust_stations.head())

# Statuts QC observationnels (valide / a_verifier / suspecte)
STATUS_ORDER = ["valide", "a_verifier", "suspecte"]
STATUS_COLORS = {"valide": "#2E7D32", "a_verifier": "#F9A825", "suspecte": "#C62828"}

qc_flagged_path = None
for candidate in [
    PROCESSED_DIR / "opensensemap_measurements_qc_flagged.parquet",
    PROCESSED_DIR / "opensensemap_measurements_qc_flagged.csv",
]:
    if candidate.exists():
        qc_flagged_path = candidate
        break

qc_status_obs = None
if qc_flagged_path is not None:
    if qc_flagged_path.suffix.lower() == ".parquet":
        qc_status_obs = pd.read_parquet(
            qc_flagged_path, columns=["qc_status", "variable", "station_id"]
        )
    else:
        print("Chargement CSV flaggé (colonnes statut seulement)...")
        qc_status_obs = pd.read_csv(
            qc_flagged_path, usecols=["qc_status", "variable", "station_id"]
        )
    print("Observations QC statut :", len(qc_status_obs), "| source:", qc_flagged_path.name)
else:
    print("Fichier flaggé absent : figures statut QC limitées aux taux des métriques.")


## 0. Contrôle rapide — exports mémoire déjà présents ?

Si `reports/tables/` et `reports/figures/` sont déjà remplis, vous pouvez **relire** les CSV/PNG sans tout régénérer.  
Sinon, exécuter les cellules suivantes jusqu’au manifeste.


In [ ]:
tables_present = sorted(TABLES_DIR.glob("*.csv"))
figures_present = sorted(FIGURES_DIR.glob("*.png"))

print(f"Tableaux : {len(tables_present)} dans {TABLES_DIR}")
for p in tables_present:
    print("  ", p.name)
print(f"Figures  : {len(figures_present)} dans {FIGURES_DIR}")
for p in figures_present:
    print("  ", p.name)

if tables_present and figures_present:
    print("\nExports déjà présents — relancer ce notebook régénère tableaux + figures.")
else:
    print("\nExécuter NB4 jusqu'au manifeste pour produire les livrables mémoire.")


## 2. Tableau 1 — Synthèse du corpus et du protocole

In [ ]:
def _median_completeness(df):
    if "completeness_rate" in df.columns:
        return float(df["completeness_rate"].median())
    if "completeness_percent" in df.columns:
        return float(df["completeness_percent"].median() / 100.0)
    if "dim_completeness" in df.columns:
        return float(df["dim_completeness"].median())
    return float("nan")

corpus = pd.DataFrame([
    {"indicateur": "Nombre de stations", "valeur": int(trust_stations["station_id"].nunique())},
    {"indicateur": "Nombre de séries station–variable", "valeur": int(len(trust_series))},
    {"indicateur": "Variables", "valeur": ", ".join(sorted(trust_series["variable"].astype(str).unique()))},
    {"indicateur": "Observations QC (somme)", "valeur": int(metrics["observations_qc"].sum())},
    {
        "indicateur": "Complétude médiane (%)",
        "valeur": round(100 * _median_completeness(metrics), 2),
    },
    {
        "indicateur": "Score de confiance médian",
        "valeur": round(float(trust_series["trust_score"].median()), 4),
    },
    {
        "indicateur": "Score de confiance (Q1–Q3)",
        "valeur": (
            f"{trust_series['trust_score'].quantile(0.25):.4f} – "
            f"{trust_series['trust_score'].quantile(0.75):.4f}"
        ),
    },
    {
        "indicateur": "QC spatial activé",
        "valeur": bool(qc_params.get("run_spatial_qc", False)),
    },
])

display(corpus)
corpus.to_csv(TABLES_DIR / "table1_synthese_corpus.csv", index=False, encoding="utf-8")
print("Export :", TABLES_DIR / "table1_synthese_corpus.csv")


## 3. Tableau 2 — Indicateurs de qualité par variable

In [ ]:
quality_by_variable = (
    metrics.groupby("variable", dropna=False)
    .agg(
        n_series=("station_id", "nunique"),
        observations=("observations_qc", "sum"),
        completeness_median=("completeness_rate", "median"),
        physical_anomaly_median=("physical_anomaly_rate", "median"),
        stuck_median=("stuck_value_rate", "median"),
        jump_median=("temporal_jump_rate", "median"),
        statistical_anomaly_median=("statistical_anomaly_rate", "median"),
        valid_rate_median=("valid_observation_rate", "median"),
        suspect_rate_median=("suspect_observation_rate", "median"),
    )
    .reset_index()
    .sort_values("variable")
)

display(quality_by_variable)
quality_by_variable.to_csv(
    TABLES_DIR / "table2_qualite_par_variable.csv",
    index=False,
    encoding="utf-8",
)
print("Export :", TABLES_DIR / "table2_qualite_par_variable.csv")


## 3bis. Figures QC — `valide` / `a_verifier` / `suspecte`

Ces statuts viennent du **Notebook 2** (synthèse des drapeaux), au niveau **observation**.  
À ne pas confondre avec les classes *fit for purpose* du Notebook 3 (niveau série).


In [ ]:
if qc_status_obs is None:
    raise FileNotFoundError(
        "Fichier opensensemap_measurements_qc_flagged.parquet/csv requis "
        "pour les figures de statut QC."
    )

status_counts = (
    qc_status_obs["qc_status"]
    .value_counts()
    .reindex(STATUS_ORDER)
    .fillna(0)
    .astype(int)
)
status_prop = status_counts / status_counts.sum()

status_summary = pd.DataFrame({
    "qc_status": status_counts.index,
    "n_observations": status_counts.values,
    "proportion": status_prop.values,
})
display(status_summary)
status_summary.to_csv(
    TABLES_DIR / "table2b_qc_status_global.csv", index=False, encoding="utf-8"
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
colors = [STATUS_COLORS[s] for s in STATUS_ORDER]
axes[0].bar(STATUS_ORDER, status_counts.values, color=colors, edgecolor="black", linewidth=0.4)
axes[0].set_ylabel("Nombre d'observations")
axes[0].set_title("Statut QC des observations")
for i, v in enumerate(status_counts.values):
    axes[0].text(i, v, f"{v:,}".replace(",", " "), ha="center", va="bottom", fontsize=8)

axes[1].pie(
    status_counts.values,
    labels=[f"{s}\n({p:.1%})" for s, p in zip(STATUS_ORDER, status_prop.values)],
    colors=colors,
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1},
)
axes[1].set_title("Répartition globale des statuts QC")
plt.tight_layout()
fig2b = FIGURES_DIR / "fig2b_qc_status_global.png"
fig.savefig(fig2b, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig2b)


In [ ]:
ct = (
    pd.crosstab(qc_status_obs["variable"], qc_status_obs["qc_status"])
    .reindex(columns=STATUS_ORDER, fill_value=0)
)
ct_prop = ct.div(ct.sum(axis=1), axis=0)
display(ct)
ct.to_csv(TABLES_DIR / "table2c_qc_status_par_variable.csv", encoding="utf-8")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
x = np.arange(len(ct_prop.index))
bottom = np.zeros(len(x))
for status in STATUS_ORDER:
    axes[0].bar(
        x, ct_prop[status].values, bottom=bottom,
        label=status, color=STATUS_COLORS[status],
    )
    bottom += ct_prop[status].values
axes[0].set_xticks(x)
axes[0].set_xticklabels(ct_prop.index)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Proportion")
axes[0].set_title("Statut QC par variable")
axes[0].legend(fontsize=8)

im = axes[1].imshow(ct.values, aspect="auto", cmap="YlOrRd")
axes[1].set_xticks(range(len(STATUS_ORDER)))
axes[1].set_xticklabels(STATUS_ORDER)
axes[1].set_yticks(range(len(ct.index)))
axes[1].set_yticklabels(ct.index)
axes[1].set_title("Effectifs observation × statut")
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        axes[1].text(
            j, i, f"{int(ct.values[i, j]):,}".replace(",", " "),
            ha="center", va="center", fontsize=8,
        )
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
fig2c = FIGURES_DIR / "fig2c_qc_status_par_variable.png"
fig.savefig(fig2c, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig2c)


In [ ]:
metrics_status = metrics.copy()
metrics_status["a_verifier_rate"] = (
    1.0
    - metrics_status["valid_observation_rate"]
    - metrics_status["suspect_observation_rate"]
).clip(lower=0)

rate_by_var = (
    metrics_status.groupby("variable")
    .agg(
        valide=("valid_observation_rate", "median"),
        a_verifier=("a_verifier_rate", "median"),
        suspecte=("suspect_observation_rate", "median"),
    )
    .reindex(columns=STATUS_ORDER)
)
display(rate_by_var)
rate_by_var.to_csv(
    TABLES_DIR / "table2d_qc_status_taux_median_par_variable.csv", encoding="utf-8"
)

fig, ax = plt.subplots(figsize=(9, 4.8))
x = np.arange(len(rate_by_var.index))
width = 0.25
for i, status in enumerate(STATUS_ORDER):
    ax.bar(
        x + i * width,
        rate_by_var[status].values,
        width=width,
        label=status,
        color=STATUS_COLORS[status],
        edgecolor="black",
        linewidth=0.3,
    )
ax.set_xticks(x + width)
ax.set_xticklabels(rate_by_var.index)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Taux médian (par série)")
ax.set_title("Taux médians valide / à vérifier / suspecte par variable")
ax.legend()
ax.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
fig2d = FIGURES_DIR / "fig2d_qc_status_taux_par_variable.png"
fig.savefig(fig2d, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig2d)


## 4. Tableau 3 — Dimensions de confiance et score global

In [ ]:
dim_cols = [
    c for c in [
        "dim_completeness", "dim_physical", "dim_stability",
        "dim_statistical", "dim_continuity", "dim_metadata",
        "dim_spatial", "trust_score",
    ] if c in trust_series.columns
]

trust_by_variable = (
    trust_series.groupby("variable")[dim_cols]
    .median()
    .reset_index()
)

display(trust_by_variable)
trust_by_variable.to_csv(
    TABLES_DIR / "table3_dimensions_confiance_par_variable.csv",
    index=False,
    encoding="utf-8",
)

# Compatibilité JSON NB3 / script
raw_weights = trust_params.get("trust_weights") or trust_params.get("weights") or {}
weights = pd.DataFrame(
    [{"dimension": k, "poids": v} for k, v in raw_weights.items()]
)
display(Markdown("**Pondérations utilisées (reproductibilité)**"))
display(weights)
weights.to_csv(TABLES_DIR / "table3b_poids_confiance.csv", index=False, encoding="utf-8")


## 5. Tableau 4 — Classification *fit for purpose*

In [ ]:
ffp_order = [
    "usage_scientifique_restreint",
    "comparaison_relative",
    "suivi_tendances",
    "exploratoire",
    "non_recommande",
]

ffp_series = ffp_summary.copy()
ffp_series["classe"] = pd.Categorical(
    ffp_series["classe"], categories=ffp_order, ordered=True
)
ffp_series = ffp_series.sort_values("classe")

ffp_stations = (
    trust_stations["fit_for_purpose_station"]
    .value_counts()
    .reindex(ffp_order)
    .fillna(0)
    .astype(int)
    .rename_axis("classe")
    .reset_index(name="n_stations")
)
ffp_stations["proportion"] = ffp_stations["n_stations"] / ffp_stations["n_stations"].sum()

display(Markdown("**Au niveau des séries**"))
display(ffp_series)
display(Markdown("**Au niveau des stations** (classe = la plus restrictive parmi les variables)"))
display(ffp_stations)

ffp_series.to_csv(TABLES_DIR / "table4a_ffp_series.csv", index=False, encoding="utf-8")
ffp_stations.to_csv(TABLES_DIR / "table4b_ffp_stations.csv", index=False, encoding="utf-8")


## 6. Tableau 5 — Stations : meilleurs et plus faibles scores

In [ ]:
top_stations = (
    trust_stations
    .sort_values("trust_score_median", ascending=False)
    [["station_name", "trust_score_median", "trust_score_min", "fit_for_purpose_station"]]
    .head(10)
    .reset_index(drop=True)
)

bottom_stations = (
    trust_stations
    .sort_values("trust_score_median", ascending=True)
    [["station_name", "trust_score_median", "trust_score_min", "fit_for_purpose_station"]]
    .head(10)
    .reset_index(drop=True)
)

display(Markdown("**Dix stations au score médian le plus élevé**"))
display(top_stations)
display(Markdown("**Dix stations au score médian le plus faible**"))
display(bottom_stations)

top_stations.to_csv(TABLES_DIR / "table5a_stations_top.csv", index=False, encoding="utf-8")
bottom_stations.to_csv(TABLES_DIR / "table5b_stations_bottom.csv", index=False, encoding="utf-8")


## 7. Figure 1 — Distribution des scores de confiance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(
    trust_series["trust_score"].dropna(),
    bins=20,
    edgecolor="black",
    color="#4C6A80",
)
axes[0].axvline(
    trust_series["trust_score"].median(),
    color="#B85C38",
    linestyle="--",
    label=f"Médiane = {trust_series['trust_score'].median():.3f}",
)
axes[0].set_xlabel("Score de confiance")
axes[0].set_ylabel("Nombre de séries")
axes[0].set_title("Distribution des scores (séries)")
axes[0].legend()

for variable, sub in trust_series.groupby("variable"):
    axes[1].hist(
        sub["trust_score"].dropna(),
        bins=15,
        alpha=0.45,
        label=variable,
        edgecolor="black",
    )
axes[1].set_xlabel("Score de confiance")
axes[1].set_ylabel("Nombre de séries")
axes[1].set_title("Distribution par variable")
axes[1].legend()

plt.tight_layout()
fig1 = FIGURES_DIR / "fig1_distribution_trust_scores.png"
fig.savefig(fig1, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig1)


## 8. Figure 2 — Profil moyen des dimensions de confiance

In [ ]:
profile_cols = [
    c for c in [
        "dim_completeness", "dim_physical", "dim_stability",
        "dim_statistical", "dim_continuity", "dim_metadata",
    ] if c in trust_series.columns
]

profile = trust_series.groupby("variable")[profile_cols].mean()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(profile_cols))
width = 0.25
for i, (variable, row) in enumerate(profile.iterrows()):
    ax.bar(x + i * width, row.values, width=width, label=variable)

ax.set_xticks(x + width)
ax.set_xticklabels(
    [c.replace("dim_", "") for c in profile_cols],
    rotation=30,
    ha="right",
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score dimensionnel moyen")
ax.set_title("Profil des dimensions de confiance par variable")
ax.legend()
ax.axhline(0.8, color="gray", linestyle=":", linewidth=1)

plt.tight_layout()
fig2 = FIGURES_DIR / "fig2_profil_dimensions.png"
fig.savefig(fig2, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig2)


## 9. Figures 3a–3c — Classification *fit for purpose* (approfondie)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

series_counts = (
    trust_series["fit_for_purpose"]
    .value_counts()
    .reindex(ffp_order)
    .fillna(0)
)
station_counts = (
    trust_stations["fit_for_purpose_station"]
    .value_counts()
    .reindex(ffp_order)
    .fillna(0)
)

axes[0].barh(series_counts.index.astype(str), series_counts.values, color="#4C6A80")
axes[0].set_xlabel("Nombre de séries")
axes[0].set_title("Fit-for-purpose (séries)")
axes[0].invert_yaxis()
for i, v in enumerate(series_counts.values):
    axes[0].text(v + 0.3, i, f"{int(v)}", va="center", fontsize=9)

axes[1].barh(station_counts.index.astype(str), station_counts.values, color="#B85C38")
axes[1].set_xlabel("Nombre de stations")
axes[1].set_title("Fit-for-purpose (stations)")
axes[1].invert_yaxis()
for i, v in enumerate(station_counts.values):
    axes[1].text(v + 0.2, i, f"{int(v)}", va="center", fontsize=9)

plt.tight_layout()
fig3a = FIGURES_DIR / "fig3a_fit_for_purpose_effectifs.png"
fig.savefig(fig3a, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig3a)



### Figure 3b — Classes FFP par variable (T / H / P)


In [ ]:
ct = (
    pd.crosstab(trust_series["variable"], trust_series["fit_for_purpose"])
    .reindex(columns=ffp_order, fill_value=0)
)
ct_prop = ct.div(ct.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

bottom = np.zeros(len(ct_prop))
colors = ["#1B4F72", "#2E86AB", "#5DADE2", "#F4A261", "#C0392B"]
x = np.arange(len(ct_prop.index))
for col, color in zip(ffp_order, colors):
    vals = ct_prop[col].values if col in ct_prop.columns else np.zeros(len(x))
    axes[0].bar(x, vals, bottom=bottom, label=col, color=color)
    bottom += vals
axes[0].set_xticks(x)
axes[0].set_xticklabels(ct_prop.index)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Proportion")
axes[0].set_title("Répartition FFP par variable")
axes[0].legend(fontsize=7, loc="upper right")

im = axes[1].imshow(ct.values, aspect="auto", cmap="YlGnBu")
axes[1].set_xticks(range(len(ct.columns)))
axes[1].set_xticklabels(ct.columns, rotation=35, ha="right", fontsize=8)
axes[1].set_yticks(range(len(ct.index)))
axes[1].set_yticklabels(ct.index)
axes[1].set_title("Effectifs série × classe")
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        axes[1].text(j, i, int(ct.values[i, j]), ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
fig3b = FIGURES_DIR / "fig3b_ffp_par_variable.png"
fig.savefig(fig3b, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig3b)

ct.to_csv(TABLES_DIR / "table4c_ffp_par_variable.csv", encoding="utf-8")
print("Export :", TABLES_DIR / "table4c_ffp_par_variable.csv")


### Figure 3c — Score de confiance selon la classe FFP


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

data_box = [
    trust_series.loc[trust_series["fit_for_purpose"] == c, "trust_score"].dropna()
    for c in ffp_order
]
bp = axes[0].boxplot(
    data_box,
    labels=[c.replace("_", "\n") for c in ffp_order],
    patch_artist=True,
    showfliers=True,
)
for patch, color in zip(bp["boxes"], ["#1B4F72", "#2E86AB", "#5DADE2", "#F4A261", "#C0392B"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[0].set_ylabel("trust_score")
axes[0].set_title("Score selon la classe FFP")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis="x", labelsize=7)

ffp_rank = {c: i for i, c in enumerate(ffp_order)}
plot = trust_series.copy()
plot["ffp_y"] = plot["fit_for_purpose"].map(ffp_rank)
for variable, sub in plot.groupby("variable"):
    axes[1].scatter(
        sub["trust_score"],
        sub["ffp_y"] + np.random.default_rng(42).uniform(-0.15, 0.15, len(sub)),
        alpha=0.7,
        s=36,
        label=variable,
    )
axes[1].set_yticks(range(len(ffp_order)))
axes[1].set_yticklabels(ffp_order, fontsize=8)
axes[1].set_xlabel("trust_score")
axes[1].set_title("Séries : score × classe (jitter)")
axes[1].legend(fontsize=8)
axes[1].set_xlim(0, 1.02)
axes[1].grid(True, axis="x", alpha=0.25)

plt.tight_layout()
fig3c = FIGURES_DIR / "fig3c_score_par_classe_ffp.png"
fig.savefig(fig3c, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig3c)


## 9bis. Figure 6 — Profil dimensionnel moyen par classe FFP


In [ ]:
dim_profile_cols = [
    c for c in [
        "dim_completeness", "dim_physical", "dim_stability",
        "dim_statistical", "dim_continuity", "dim_metadata",
    ] if c in trust_series.columns
]

profile_ffp = (
    trust_series.groupby("fit_for_purpose")[dim_profile_cols]
    .mean()
    .reindex(ffp_order)
)

fig, ax = plt.subplots(figsize=(11, 5.2))
im = ax.imshow(profile_ffp.values, aspect="auto", cmap="RdYlGn", vmin=0.5, vmax=1.0)
ax.set_xticks(range(len(dim_profile_cols)))
ax.set_xticklabels([c.replace("dim_", "") for c in dim_profile_cols], rotation=30, ha="right")
ax.set_yticks(range(len(profile_ffp.index)))
ax.set_yticklabels(profile_ffp.index)
ax.set_title("Profil moyen des dimensions par classe FFP")
for i in range(profile_ffp.shape[0]):
    for j in range(profile_ffp.shape[1]):
        val = profile_ffp.values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Score moyen")
plt.tight_layout()
fig6 = FIGURES_DIR / "fig6_profil_dimensions_par_classe.png"
fig.savefig(fig6, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig6)

profile_ffp.to_csv(
    TABLES_DIR / "table4d_profil_dimensions_par_classe.csv",
    encoding="utf-8",
)


## 9ter. Figure 7 — Taux d’anomalies QC par variable


In [ ]:
rate_cols = [
    c for c in [
        "physical_anomaly_rate", "stuck_value_rate", "temporal_jump_rate",
        "statistical_anomaly_rate", "gap_event_rate", "unit_unknown_rate",
    ] if c in metrics.columns
]

rates = metrics.groupby("variable")[rate_cols].median()
fig, ax = plt.subplots(figsize=(10.5, 4.8))
im = ax.imshow(rates.values, aspect="auto", cmap="OrRd", vmin=0, vmax=max(0.05, float(rates.values.max())))
ax.set_xticks(range(len(rate_cols)))
ax.set_xticklabels([c.replace("_rate", "").replace("_", "\n") for c in rate_cols], fontsize=8)
ax.set_yticks(range(len(rates.index)))
ax.set_yticklabels(rates.index)
ax.set_title("Médiane des taux d'anomalies QC par variable")
for i in range(rates.shape[0]):
    for j in range(rates.shape[1]):
        ax.text(j, i, f"{rates.values[i, j]:.3f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Taux médian")
plt.tight_layout()
fig7 = FIGURES_DIR / "fig7_taux_anomalies_par_variable.png"
fig.savefig(fig7, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig7)


## 9quater. Figure 8 — Stabilité vs plausibilité (colorée par classe)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors_map = {
    "usage_scientifique_restreint": "#1B4F72",
    "comparaison_relative": "#2E86AB",
    "suivi_tendances": "#5DADE2",
    "exploratoire": "#F4A261",
    "non_recommande": "#C0392B",
}
for classe in ffp_order:
    sub = trust_series[trust_series["fit_for_purpose"] == classe]
    if sub.empty:
        continue
    ax.scatter(
        sub["dim_physical"],
        sub["dim_stability"],
        c=colors_map.get(classe, "gray"),
        label=classe,
        alpha=0.8,
        s=45,
        edgecolor="white",
        linewidth=0.4,
    )
ax.axvline(0.97, color="gray", linestyle=":", linewidth=1)
ax.axhline(0.90, color="gray", linestyle=":", linewidth=1)
ax.set_xlabel("dim_physical")
ax.set_ylabel("dim_stability")
ax.set_title("Stabilité vs plausibilité (seuils usage scientifique)")
ax.set_xlim(0.5, 1.02)
ax.set_ylim(0.4, 1.02)
ax.legend(fontsize=7, loc="lower left")
ax.grid(True, alpha=0.25)
plt.tight_layout()
fig8 = FIGURES_DIR / "fig8_stabilite_vs_physique_ffp.png"
fig.savefig(fig8, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig8)


## 11bis. Figure 5b — Carte des stations par classe FFP


## 10. Figure 4 — Complétude vs score de confiance

In [ ]:
plot_df = trust_series.copy()
if "completeness_rate" not in plot_df.columns and "dim_completeness" in plot_df.columns:
    plot_df["completeness_rate"] = plot_df["dim_completeness"]

fig, ax = plt.subplots(figsize=(7.5, 5))
for variable, sub in plot_df.groupby("variable"):
    ax.scatter(
        sub["completeness_rate"],
        sub["trust_score"],
        alpha=0.75,
        label=variable,
        s=40,
    )

ax.set_xlabel("Taux de complétude")
ax.set_ylabel("Score de confiance")
ax.set_title("Relation complétude – confiance")
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
fig4 = FIGURES_DIR / "fig4_completude_vs_confiance.png"
fig.savefig(fig4, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig4)


## 11. Figure 5 — Carte des stations (score médian)

In [ ]:
geo = trust_stations.merge(
    metrics.groupby("station_id", as_index=False).agg(
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
    ),
    on="station_id",
    how="left",
)

fig, ax = plt.subplots(figsize=(7.5, 6))
sc = ax.scatter(
    geo["longitude"],
    geo["latitude"],
    c=geo["trust_score_median"],
    cmap="viridis",
    s=55,
    edgecolor="black",
    linewidth=0.4,
)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Score de confiance médian")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Répartition spatiale des stations et confiance médiane")
ax.grid(True, alpha=0.25)

plt.tight_layout()
fig5 = FIGURES_DIR / "fig5_carte_stations_confiance.png"
fig.savefig(fig5, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig5)


In [ ]:
geo_ffp = trust_stations.merge(
    metrics.groupby("station_id", as_index=False).agg(
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
    ),
    on="station_id",
    how="left",
)

colors_map = {
    "usage_scientifique_restreint": "#1B4F72",
    "comparaison_relative": "#2E86AB",
    "suivi_tendances": "#5DADE2",
    "exploratoire": "#F4A261",
    "non_recommande": "#C0392B",
}

fig, ax = plt.subplots(figsize=(8, 6.2))
for classe in ffp_order:
    sub = geo_ffp[geo_ffp["fit_for_purpose_station"] == classe]
    if sub.empty:
        continue
    ax.scatter(
        sub["longitude"],
        sub["latitude"],
        c=colors_map.get(classe, "gray"),
        label=f"{classe} (n={len(sub)})",
        s=60,
        edgecolor="black",
        linewidth=0.35,
        alpha=0.9,
    )
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Stations selon la classe fit-for-purpose")
ax.legend(fontsize=7, loc="best")
ax.grid(True, alpha=0.25)
plt.tight_layout()
fig5b = FIGURES_DIR / "fig5b_carte_stations_ffp.png"
fig.savefig(fig5b, dpi=200, bbox_inches="tight")
plt.show()
print("Figure :", fig5b)


## 12. Diagnostic croisé : pourquoi une station bien scorée peut rester « exploratoire »

In [ ]:
examples = trust_stations[
    (trust_stations["trust_score_median"] >= 0.95)
    & (trust_stations["fit_for_purpose_station"].isin(["exploratoire", "non_recommande", "suivi_tendances"]))
].copy()

if examples.empty:
    examples = trust_stations.sort_values("trust_score_median", ascending=False).head(3)

detail_rows = []
for _, station in examples.head(5).iterrows():
    sub = trust_series[trust_series["station_id"] == station["station_id"]].copy()
    for _, row in sub.iterrows():
        detail_rows.append({
            "station_name": station["station_name"],
            "variable": row["variable"],
            "trust_score": row["trust_score"],
            "fit_for_purpose": row["fit_for_purpose"],
            "dim_physical": row.get("dim_physical", np.nan),
            "dim_stability": row.get("dim_stability", np.nan),
            "dim_completeness": row.get("dim_completeness", np.nan),
            "classe_station": station["fit_for_purpose_station"],
        })

diagnostic = pd.DataFrame(detail_rows)
display(diagnostic)
diagnostic.to_csv(
    TABLES_DIR / "table6_diagnostic_classe_restrictive.csv",
    index=False,
    encoding="utf-8",
)

display(Markdown(
    "Interprétation : la classe station retient la contrainte **la plus restrictive** "
    "parmi les variables. Un score médian élevé peut coexister avec une classe modeste "
    "si une seule variable échoue aux seuils *fit for purpose*."
))


## 13. Export groupé pour le mémoire

In [ ]:
manifest = pd.DataFrame([
    {"type": "table", "fichier": str(p.relative_to(PROJECT_ROOT))}
    for p in sorted(TABLES_DIR.glob("*.csv"))
] + [
    {"type": "figure", "fichier": str(p.relative_to(PROJECT_ROOT))}
    for p in sorted(FIGURES_DIR.glob("*.png"))
])

manifest_path = MANIFEST_CSV
manifest.to_csv(manifest_path, index=False, encoding="utf-8")
display(manifest)
print("Manifeste :", manifest_path)
print(f"Tableaux  : {len(list(TABLES_DIR.glob('*.csv')))} → {TABLES_DIR}")
print(f"Figures   : {len(list(FIGURES_DIR.glob('*.png')))} → {FIGURES_DIR}")
print("\nChaîne empirique NB1→NB4 prête pour la rédaction.")


## 14. Bilan pour la rédaction

Éléments empiriques prêts pour le chapitre Résultats :

- description du corpus ;
- indicateurs de qualité par variable ;
- dimensions et score de confiance ;
- classification *fit for purpose* ;
- figures reproductibles ;
- diagnostic classe restrictive vs score médian.

### Points à rappeler dans la discussion

1. le cadre évalue une **confiance relative**, non une exactitude absolue ;
2. les seuils et poids sont des **choix méthodologiques** documentés ;
3. l’absence de référence WMO colocalisée limite la validation externe.

### Pipeline complet

Si cette étape est terminée, la chaîne empirique NB1→NB4 est bouclée.  
Reprendre ensuite la rédaction : revue de littérature → cadre méthodologique → résultats → discussion.
